In [1]:
%%writefile requirements.txt
fastapi
uvicorn

Writing requirements.txt


In [2]:
%%writefile main.py
from fastapi import FastAPI
from pydantic import BaseModel

# 1. Inicializamos la aplicación FastAPI
app = FastAPI(
    title="Mi Primera API en el Reto",
    description="API simple explicada paso a paso con FastAPI, Docker y Ngrok",
    version="1.0.0"
)

# Base de datos simulada (una lista en memoria)
items_db = ["Ordenador", "Teclado", "Ratón"]

# Modelo de datos para validar cuando alguien añade un producto nuevo
class NuevoItem(BaseModel):
    nombre: str

# MÉTODO 1: Ruta principal de bienvenida (GET)
@app.get("/", summary="Saludo inicial")
def inicio():
    return {"mensaje": "¡Hola! La API está funcionando correctamente."}

# MÉTODO 2: Consultar todos los elementos de la lista (GET)
@app.get("/items", summary="Obtener lista de objetos")
def obtener_items():
    return {"total": len(items_db), "items": items_db}

# MÉTODO 3: Añadir un elemento nuevo a la lista (POST)
@app.post("/items", summary="Añadir un objeto nuevo")
def agregar_item(item: NuevoItem):
    items_db.append(item.nombre)
    return {"mensaje": f"Se ha añadido '{item.nombre}' con éxito.", "lista_actualizada": items_db}

Writing main.py


In [3]:
%%writefile Dockerfile
# 1. Usamos una versión ligera oficial de Python
FROM python:3.10-slim

# 2. Creamos y nos movemos a la carpeta de trabajo dentro del contenedor
WORKDIR /app

# 3. Copiamos la lista de librerías e instalamos todo
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# 4. Copiamos el código de nuestra API dentro del contenedor
COPY main.py .

# 5. Exponemos el puerto 8000 (la puerta por donde entrarán las peticiones)
EXPOSE 8000

# 6. Comando final para encender el servidor al arrancar el contenedor
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]

Writing Dockerfile


In [4]:
!pip install -q fastapi uvicorn pyngrok

In [6]:
import subprocess
import time
from pyngrok import ngrok

# 1. Pon aquí tu token de Ngrok (entre las comillas)
NGROK_TOKEN = "3Lrok8G6JOCroC7fpQI22BsjcId_6fRmCK2GBvkF6Z6nn8L5o"
ngrok.set_auth_token(NGROK_TOKEN)

# 2. Cerramos túneles previos por si ejecutas la celda varias veces
ngrok.kill()

# 3. Arrancamos la API en segundo plano en el puerto 8000
proceso_api = subprocess.Popen(["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(3) # Esperamos 3 segundos a que arranque bien

# 4. Abrimos el túnel de Ngrok hacia el puerto 8000
tunel_publico = ngrok.connect(8000)

print("==========================================")
print(f"🌍 URL Pública de tu API: {tunel_publico.public_url}")
print(f"📑 Entra aquí para ver SWAGGER: {tunel_publico.public_url}/docs")
print("==========================================")

🌍 URL Pública de tu API: https://surviving-streak-passover.ngrok-free.dev
📑 Entra aquí para ver SWAGGER: https://surviving-streak-passover.ngrok-free.dev/docs
